In [1]:
import requests
import os
from dotenv import load_dotenv
from langchain.agents import AgentExecutor, create_tool_calling_agent
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain.prompts import ChatPromptTemplate

load_dotenv()

x_rapidapi_key = os.getenv("x_rapidapi_key")
x_rapidapi_host = os.getenv("x_rapidapi_host")

headers = {
	"x-rapidapi-key": x_rapidapi_key,
	"x-rapidapi-host": x_rapidapi_host
}
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")

# from langchain_openai import ChatOpenAI

# llm = ChatOpenAI(model="gpt-3.5-turbo")

llm = ChatOpenAI(model="gpt-4o")

In [2]:

from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o")

response = llm.invoke("hey how are you?")

print(response.content)

Hello! I'm here and ready to help. How can I assist you today?


In [ ]:
# Get cheapest flight details



In [2]:
# # Get available language
# url = "https://google-flights2.p.rapidapi.com/api/v1/getLanguages"

# response = requests.get(url, headers=headers)

# data = response.json()

# for language in data["data"]:
#     print(language)

In [3]:
# # Get available language
# url = "https://google-flights2.p.rapidapi.com/api/v1/getCurrency"

# response = requests.get(url, headers=headers)

# data = response.json()

# for currency in data["data"]:
#     print(currency)

In [4]:
# # Get available language
# url = "https://google-flights2.p.rapidapi.com/api/v1/getLocations"

# response = requests.get(url, headers=headers)

# data = response.json()

# for currency in data["data"]:
#     print(currency)

In [34]:
# @tool
def get_flight_information(querystring: str) -> dict:
    """
    Fetches flight information based on the querystring.
    """
    url = "https://google-flights2.p.rapidapi.com/api/v1/searchFlights"
    response = requests.get(url, headers=headers, params=querystring)
    data = response.json()

    flight_details = {
        "total_flights": len(data['data']['itineraries']['topFlights']),
        "flights": {}
    }

    for index, flight in enumerate(data['data']['itineraries']['topFlights']):
        flight_info = flight['flights'][0]
        flight_details["flights"][str(index)] = {
            "Aircraft": flight_info.get('aircraft', 'N/A'),
            "Flight Number": flight_info.get('flight_number', 'N/A'),
            "Departure Time": flight.get('departure_time', 'N/A'),
            "Arrival Time": flight.get('arrival_time', 'N/A'),
            "Price": f"${flight.get('price', 'N/A')}",
            "Duration": flight.get('duration', {}).get('text', 'N/A')
        }
    return flight_details

In [36]:
querystring = {"departure_id":"LAX","arrival_id":"JFK","outbound_date":"2025-02-12","return_date":"2025-02-15","travel_class":"ECONOMY","adults":"1","show_hidden":"0","currency":"USD"}

response = get_flight_information(querystring)

{'total_flights': 3,
 'flights': {'0': {'Aircraft': 'Airbus A321 (Sharklets)',
   'Flight Number': 'AA 118',
   'Departure Time': '12-02-2025 06:12 AM',
   'Arrival Time': '12-02-2025 02:36 PM',
   'Price': '$717',
   'Duration': '5 hr 24 min'},
  '1': {'Aircraft': 'Airbus A320',
   'Flight Number': 'B6 224',
   'Departure Time': '12-02-2025 07:05 AM',
   'Arrival Time': '12-02-2025 03:30 PM',
   'Price': '$717',
   'Duration': '5 hr 25 min'},
  '2': {'Aircraft': 'Boeing 767',
   'Flight Number': 'DL 934',
   'Departure Time': '12-02-2025 06:00 AM',
   'Arrival Time': '12-02-2025 02:20 PM',
   'Price': '$717',
   'Duration': '5 hr 20 min'}}}

In [30]:
prompt = ChatPromptTemplate.from_template("""
You are a helpful travel itinerary planner. Your task involves two steps:

**Step 1: Generate Query String**
- Extract information from the user request to create a query string for flight search.
- **Important:** Return the query string as a Python dictionary (NOT as a JSON string or within quotes).

**Step 2: Analyze Flight Information**
- After receiving the flight data, analyze the options to suggest the best flights based on price, duration, and convenience.

*Example Query (Correct Format):*
{{
    "departure_id": "DAC",
    "arrival_id": "CGP",
    "outbound_date": "2025-02-12",
    "return_date": "2025-02-15",
    "travel_class": "ECONOMY",
    "adults": "1",
    "show_hidden": "0",
    "currency": "USD"
}}

**Instructions:**
- Convert **departure city** to airport code (`departure_id`).
- Convert **arrival city** to airport code (`arrival_id`).
- Identify **outbound date** and **return date**.
- Determine **travel class** (e.g., Economy, Business).
- Identify number of **adults** traveling.
- Always set `show_hidden` to "0" and `currency` to "USD".

**Flight Analysis Criteria:**
- Highlight the **cheapest flight**.
- Recommend the **fastest flight**.
- Suggest the **best overall option** (considering price, duration, and time).

**User Request:**  
{question}

{agent_scratchpad}
""")


In [31]:
question = "I want to go chittagong from dhaka at 12 february 2025 and return 15 february 2025. I'm a single person and willing to travel in economy class"

In [32]:
# chain = prompt | llm
# response = chain.invoke(question)
# print(response.content)

In [33]:
# Bind tools
tools = [get_flight_information]
llm_with_tools = llm.bind_tools(tools)

# ReAct agent
agent = create_tool_calling_agent(llm_with_tools, tools, prompt)

# Agent Executor
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

# Invoke the agent
response = agent_executor.invoke({"question": question})

# Displaying the output
print(response['output'])




> Entering new AgentExecutor chain...

Invoking: `get_flight_information` with `{'querystring': {'adults': '1', 'return_date': '2025-02-15', 'currency': 'USD', 'travel_class': 'ECONOMY', 'departure_id': 'DAC', 'arrival_id': 'CGP', 'outbound_date': '2025-02-12', 'show_hidden': '0'}}`




ValidationError: 1 validation error for get_flight_information
querystring
  Input should be a valid string [type=string_type, input_value={'adults': '1', 'return_d...12', 'show_hidden': '0'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.10/v/string_type